# Object detection with YOLO and OpenVINO

## Install dependencies

In [1]:
!pip install requests ultralytics openvino nncf moviepy --extra-index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu


## Get video

In [2]:
from IPython.display import display, Image
import cv2
from utils import download_video, run_inference_save_mp4, run_inference_on_image, run_live_inference, display_video

# Download the sample video
video_name = "sample_video.mp4"
video_file = f"data/{video_name}"
download_video("https://storage.openvinotoolkit.org/repositories/openvino_notebooks/data/data/video/people.mp4", video_file)

display_video(video_file)

100%|███████████████████████████████████████████████| 3.54M/3.54M [00:00<00:00, 4.06MB/s]


Download complete: data\sample_video.mp4


PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\anish\\AppData\\Local\\Temp\\tmp_d0h6vfx.gif'

## Get model

In [ ]:
import ipywidgets as widgets

# Select the model type
model_dropdown = widgets.Dropdown(
    options=["yolo11n", "yolo11s", "yolo11m", "yolo11l", "yolo11x"],
    value="yolo11n",
    description="Model:"
)
model_dropdown

In [ ]:
from ultralytics import YOLO

# Load the model
model_name = f"models/{model_dropdown.value}"
yolo_model = YOLO(model_name)

In [ ]:
# Run prediction on the video (saves directly as MP4 via OpenCV)
processed_video, results = run_inference_save_mp4(yolo_model, video_file, f"runs/detect/pytorch/{video_name}")
display_video(processed_video)

## Inference on 1 image

In [ ]:
# Extract first frame from video and run inference
cap = cv2.VideoCapture(video_file)
_, first_frame = cap.read()
cap.release()

image_path = "data/first_frame.jpg"
cv2.imwrite(image_path, first_frame)

annotated = run_inference_on_image(yolo_model, image_path)
_, buf = cv2.imencode(".jpeg", cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
display(Image(data=buf.tobytes()))

In [ ]:
import statistics as stat

# Calculate mean inference time (skip first inference which is usually longer)
avg_inference_time = stat.mean(results[1:])
print(f"One image inference time in PyTorch: {avg_inference_time:.2f}ms")

## Use OpenVINO

In [ ]:
# Convert the model to OV format with fixed input shape (640x640) and FP16 precision
ov_model_path = yolo_model.export(format="openvino", dynamic=False, half=True)

# Reload the model
ov_yolo_model = YOLO(ov_model_path, task="detect")

# Run prediction once again on the video (saves directly as MP4)
processed_video, ov_inference_times = run_inference_save_mp4(
    ov_yolo_model, video_file, f"runs/detect/openvino_cpu/{video_name}", device="intel:cpu"
)
display_video(processed_video)

In [ ]:
import statistics as stat

# Calculate mean inference time (skip first inference which is usually longer)
avg_ov_inference_time = stat.mean(ov_inference_times[1:])
print(f"One image inference time in OpenVINO on CPU: {avg_ov_inference_time:.2f}ms")

## Available devices

In [ ]:
import openvino as ov

core = ov.Core()
available_devices = core.available_devices

print(available_devices)
print([core.get_property(device, "FULL_DEVICE_NAME") for device in available_devices])

## Try other devices

In [ ]:
if "GPU" in available_devices:
    # Reload the model
    ov_yolo_model = YOLO(ov_model_path, task="detect")
    # Run inference on GPU
    _, ov_gpu_inference_times = run_inference_save_mp4(
        ov_yolo_model, video_file, f"runs/detect/openvino_gpu/{video_name}", device="intel:gpu"
    )
    # Calculate mean inference time (skip first inference which is usually longer)
    avg_ov_gpu_inference_time = stat.mean(ov_gpu_inference_times[1:])
    print(f"One image inference time in OpenVINO on GPU: {avg_ov_gpu_inference_time:.2f}ms")

In [ ]:
if "NPU" in available_devices:
    # Reload the model
    ov_yolo_model = YOLO(ov_model_path, task="detect")
    # Run inference on NPU
    _, ov_npu_inference_times = run_inference_save_mp4(
        ov_yolo_model, video_file, f"runs/detect/openvino_npu/{video_name}", device="intel:npu"
    )
    # Calculate mean inference time (skip first inference which is usually longer)
    avg_ov_npu_inference_time = stat.mean(ov_npu_inference_times[1:])
    print(f"One image inference time in OpenVINO on NPU: {avg_ov_npu_inference_time:.2f}ms")

## Quantize model

In [ ]:
# Convert and quantize the model to OV format with fixed input shape (640x640) and INT8 precision
ov_int8_model_path = yolo_model.export(format="openvino", dynamic=False, int8=True, data="coco128.yaml")

In [ ]:
import ipywidgets as widgets

# Select the model type
device_dropdown = widgets.Dropdown(
    options=available_devices,
    value="CPU",
    description="Device:"
)
device_dropdown

In [ ]:
# Load int8 model
ov_int8_yolo_model = YOLO(ov_int8_model_path, task="detect")
# Run inference on the selected device (saves directly as MP4)
processed_video_int8, ov_int8_inference_times = run_inference_save_mp4(
    ov_int8_yolo_model, video_file, f"runs/detect/openvino_int8/{video_name}",
    device=f"intel:{device_dropdown.value}"
)

# Calculate mean inference time (skip first inference which is usually longer)
avg_ov_int8_inference_time = stat.mean(ov_int8_inference_times[1:])
print(f"One image inference time in OpenVINO on {device_dropdown.value}: {avg_ov_int8_inference_time:.2f}ms")

In [ ]:
# Show the processed video (already saved as MP4)
display_video(processed_video_int8)

## Live inference

Run object detection on your webcam. **Press Q** or **close the popup window** to stop.

In [ ]:
# Live inference with webcam. Exit: press Q or close the popup window.
run_live_inference(
    ov_int8_yolo_model,
    device=f"intel:{device_dropdown.value}",
)